# Notebook 06 — Nemotron Inference Service + MLflow Tracing

**RHOAI 3.5 | Nemotron LLM | MLflow 3.13.0 Tracing API**

This notebook traces live calls to the Nemotron model deployed as an RHOAI InferenceService,
capturing prompts, responses, token usage, and latency in the RHOAI GenAI Dashboard.

1. Connect to the Nemotron inference endpoint and verify authentication
2. Trace individual completions with `@mlflow.trace`
3. Multi-turn conversation with session grouping
4. Temperature sweep — compare sampling parameters as MLflow experiment runs

---
**Prerequisites:** Notebook 01 completed (MLflow tracking URI and auth configured).
**No additional packages** — uses `requests` (pre-installed in all RHOAI workbench images).

## Section 1 — Setup & Connection

In [ ]:
import mlflow
import mlflow.tracking
from mlflow.entities import SpanType
import requests, os, time, uuid, json
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
requests.packages.urllib3.disable_warnings(
    requests.packages.urllib3.exceptions.InsecureRequestWarning
)

# ── Nemotron InferenceService ──────────────────────────────────────────
# Route exposed by RHOAI Model Serving for the Nemotron InferenceService.
LLM_BASE_URL = 'https://llm-base-url.com' #replace this url

# Model name as registered in the InferenceService.
# Verify with:  oc get inferenceservice -n <your-namespace>
# or auto-detected below via GET /v1/models
MODEL_ID = 'nemotron-3-5-lightning'   # <--- update if different

EXPERIMENT_NAME = 'rhoai-nemotron-tracing'
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'Nemotron endpoint : {LLM_BASE_URL}')
print(f'MLflow experiment : {EXPERIMENT_NAME}')
print(f'Tracking URI      : {mlflow.get_tracking_uri()}')

In [ ]:
# ── Authentication ────────────────────────────────────────────────────
# RHOAI Model Serving uses the pod's service account token for auth.
# Inside a workbench the token is mounted at the standard k8s path.
SA_TOKEN_PATH = '/var/run/secrets/kubernetes.io/serviceaccount/token'
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        INFER_TOKEN = f.read().strip()
    print('Using workbench service account token.')
else:
    # Outside a workbench: set OC_TOKEN=\$(oc whoami --show-token) before starting Jupyter
    INFER_TOKEN = os.environ.get('OC_TOKEN', '')
    if INFER_TOKEN:
        print('Using OC_TOKEN from environment.')
    else:
        print('WARNING: no token found — unauthenticated requests may be rejected.')
        print('  Run in a terminal:  export OC_TOKEN=\$(oc whoami --show-token)')

HEADERS = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {INFER_TOKEN}',
}

In [ ]:
# Auto-detect the deployed model name, verify the endpoint, and set protocol format
API_FORMAT    = 'openai'   # 'openai' | 'kserve_v2'
CHAT_ENDPOINT = f'{LLM_BASE_URL}/v1/chat/completions'

try:
    resp = requests.get(
        f'{LLM_BASE_URL}/v1/models',
        headers=HEADERS, verify=False, timeout=10,
    )
    resp.raise_for_status()
    available = [m['id'] for m in resp.json().get('data', [])]
    print(f'Inference service reachable. Models: {available}')
    if available and MODEL_ID not in available:
        MODEL_ID = available[0]
        print(f'  Auto-updated MODEL_ID → {MODEL_ID}')
    API_FORMAT    = 'openai'
    CHAT_ENDPOINT = f'{LLM_BASE_URL}/v1/chat/completions'
    print(f'Protocol: OpenAI-compatible')
except requests.exceptions.HTTPError as e:
    if e.response.status_code == 404:
        print('/v1/models → 404 — trying KServe v2 protocol...')
        try:
            r2 = requests.get(f'{LLM_BASE_URL}/v2/health/ready',
                              headers=HEADERS, verify=False, timeout=10)
            if r2.status_code == 200:
                API_FORMAT    = 'kserve_v2'
                CHAT_ENDPOINT = f'{LLM_BASE_URL}/v2/models/{MODEL_ID}/infer'
                print(f'Protocol: KServe v2  ({CHAT_ENDPOINT})')
            else:
                print(f'/v2/health/ready returned {r2.status_code}')
        except requests.exceptions.RequestException as e2:
            print(f'KServe v2 probe failed: {e2}')
    elif e.response.status_code in (401, 403):
        print(f'Auth error {e.response.status_code} — check INFER_TOKEN above.')
    else:
        raise
except requests.exceptions.ConnectionError as e:
    print(f'Connection failed: {e}')
    print('  Check: oc get route -n <namespace> | grep nemotron')

print(f'\nMODEL_ID : {MODEL_ID}')
print(f'API_FORMAT     : {API_FORMAT}')
print(f'CHAT_ENDPOINT  : {CHAT_ENDPOINT}')

## Section 2 — Single Traced Completion

The `@mlflow.trace(span_type=SpanType.LLM)` decorator automatically captures inputs,
outputs, and execution duration. Span attributes add model-specific metadata
(token counts, latency, temperature) visible in the RHOAI Dashboard trace waterfall.

In [ ]:
def _post_completion(messages, temperature=0.7, max_tokens=512):
    """Raw HTTP call — no MLflow instrumentation. Handles OpenAI and KServe v2."""
    t0 = time.time()
    if API_FORMAT == 'openai':
        resp = requests.post(
            CHAT_ENDPOINT,
            headers=HEADERS,
            json={'model': MODEL_ID, 'messages': messages,
                  'temperature': temperature, 'max_tokens': max_tokens},
            verify=False, timeout=60,
        )
        resp.raise_for_status()
        data = resp.json()
    else:   # kserve_v2
        prompt = '\n'.join(
            f"{m['role'].upper()}: {m['content']}" for m in messages
        )
        resp = requests.post(
            CHAT_ENDPOINT,
            headers=HEADERS,
            json={
                'inputs': [{'name': 'text_input', 'shape': [1],
                            'datatype': 'BYTES', 'data': [prompt]}],
                'parameters': {'temperature': temperature,
                               'max_new_tokens': max_tokens},
            },
            verify=False, timeout=60,
        )
        resp.raise_for_status()
        kv2  = resp.json()
        out  = kv2.get('outputs', [{}])
        text = out[0].get('data', [''])[0] if out else ''
        # Normalize to OpenAI shape so the rest of the code is uniform
        data = {'choices': [{'message': {'content': text}, 'finish_reason': 'stop'}],
                'usage': {}}
    data['_latency_ms'] = int((time.time() - t0) * 1000)
    return data


@mlflow.trace(span_type=SpanType.LLM, name='nemotron-chat')
def traced_completion(messages, temperature=0.7, max_tokens=512, system_prompt=None):
    """MLflow-traced completion. Captures token usage and latency as span attributes."""
    if system_prompt:
        messages = [{'role': 'system', 'content': system_prompt}] + list(messages)

    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute('llm.model',        MODEL_ID)
        span.set_attribute('llm.temperature',  temperature)
        span.set_attribute('llm.max_tokens',   max_tokens)
        span.set_attribute('llm.endpoint',     CHAT_ENDPOINT)
        span.set_attribute('llm.api_format',   API_FORMAT)

    data   = _post_completion(messages, temperature=temperature, max_tokens=max_tokens)
    usage  = data.get('usage', {})
    choice = data['choices'][0]

    finish_reason = choice.get('finish_reason', '')
    content       = choice['message'].get('content') or ''

    if span:
        span.set_attribute('llm.prompt_tokens',     usage.get('prompt_tokens', 0))
        span.set_attribute('llm.completion_tokens', usage.get('completion_tokens', 0))
        span.set_attribute('llm.total_tokens',      usage.get('total_tokens', 0))
        span.set_attribute('llm.latency_ms',        data['_latency_ms'])
        span.set_attribute('llm.finish_reason',     finish_reason)
        if finish_reason == 'tool_calls':
            span.set_attribute('llm.tool_calls', str(choice['message'].get('tool_calls', [])))

    return {
        'content'      : content,
        'finish_reason': finish_reason,
        'usage'        : usage,
        'latency_ms'   : data['_latency_ms'],
    }

print('traced_completion() ready.')

In [ ]:
SYSTEM_PROMPT = (
    'You are a helpful assistant for Red Hat OpenShift AI and MLflow. '
    'Answer questions accurately and concisely.'
)

question = 'What is the difference between an MLflow experiment and a run?'

with mlflow.start_run(run_name='single-completion'):
    mlflow.set_tags({'notebook': '06_nemotron_tracing', 'model': MODEL_ID})

    result = traced_completion(
        messages=[{'role': 'user', 'content': question}],
        temperature=0.7,
        max_tokens=300,
        system_prompt=SYSTEM_PROMPT,
    )

    mlflow.log_param('question', question)
    mlflow.log_metrics({
        'prompt_tokens'    : result['usage'].get('prompt_tokens', 0),
        'completion_tokens': result['usage'].get('completion_tokens', 0),
        'total_tokens'     : result['usage'].get('total_tokens', 0),
        'latency_ms'       : result['latency_ms'],
    })

print(f'Q: {question}')
print(f'A: {result["content"]}')
print()
print(f'Latency : {result["latency_ms"]} ms')
print(f'Tokens  : {result["usage"]}')
print()
print('View trace: RHOAI Dashboard > Experiments > rhoai-nemotron-tracing > run > Traces tab')

## Section 3 — Multi-Turn Conversation with Session Tracking

Each turn is a `mlflow.start_span` context. `span.set_attribute('mlflow.trace.session_id', ...)`
groups all turns under one session in the RHOAI Dashboard **Sessions** tab.

In [ ]:
session_id = str(uuid.uuid4())
print(f'Session ID: {session_id}')

conversation = [
    'What is RHOAI and how does it differ from plain OpenShift?',
    'How does MLflow authentication work inside an RHOAI workbench?',
    'What is trace archival and when should I enable it in production?',
]

history      = []
total_tokens = 0

with mlflow.start_run(run_name='multi-turn-nemotron'):
    mlflow.set_tags({
        'notebook'  : '06_nemotron_tracing',
        'model'     : MODEL_ID,
        'session_id': session_id,
    })

    for turn, question in enumerate(conversation, 1):
        history.append({'role': 'user', 'content': question})

        with mlflow.start_span(name=f'turn-{turn}', span_type=SpanType.CHAIN) as span:
            span.set_inputs({'turn': turn, 'question': question})
            span.set_attribute('mlflow.trace.session_id', session_id)

            result = traced_completion(
                messages=list(history),
                temperature=0.6,
                max_tokens=300,
                system_prompt=SYSTEM_PROMPT,
            )
            history.append({'role': 'assistant', 'content': result['content']})
            total_tokens += result['usage'].get('total_tokens', 0)
            span.set_outputs({'response': result['content'][:200]})

        mlflow.log_metric('cumulative_tokens', total_tokens, step=turn)
        print(f'  Turn {turn}: {question[:60]}...')
        print(f'           {result["content"][:100]}...')
        print()

print(f'Total tokens: {total_tokens}')
print(f'Session: {session_id}')

## Section 4 — Temperature Sweep as MLflow Runs

Run the same prompt at three temperatures and compare token usage and latency
in the MLflow Dashboard experiment comparison view.

In [ ]:
SWEEP_QUESTION = (
    'Explain in 2-3 sentences how MLflow model registry aliases work '
    'and why they are preferred over stage names.'
)

temperatures  = [0.1, 0.5, 0.9]
sweep_results = []

for temp in temperatures:
    with mlflow.start_run(run_name=f'nemotron-temp-{str(temp).replace(".", "_")}'):
        mlflow.set_tags({
            'notebook': '06_nemotron_tracing',
            'model'   : MODEL_ID,
            'sweep'   : 'temperature',
        })
        mlflow.log_param('temperature', temp)

        result = traced_completion(
            messages=[{'role': 'user', 'content': SWEEP_QUESTION}],
            temperature=temp,
            max_tokens=200,
            system_prompt=SYSTEM_PROMPT,
        )

        mlflow.log_metrics({
            'prompt_tokens'    : result['usage'].get('prompt_tokens', 0),
            'completion_tokens': result['usage'].get('completion_tokens', 0),
            'total_tokens'     : result['usage'].get('total_tokens', 0),
            'latency_ms'       : result['latency_ms'],
            'response_length'  : len(result['content']),
        })

        sweep_results.append({**result, 'temperature': temp})
        print(f'temp={temp}: {result["content"][:90]}...')

print()
print(f'{"Temp":<6}  {"Latency":>10}  {"Tokens":>8}  {"Length":>8}')
print('-' * 40)
for r in sweep_results:
    print(f'{r["temperature"]:<6}  {r["latency_ms"]:>8}ms  '
          f'{r["usage"].get("total_tokens",0):>7}  {len(r["content"]):>7}')

In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="tags.sweep = 'temperature'",
    order_by=['params.temperature ASC'],
)

if 'params.temperature' in runs_df.columns and 'metrics.latency_ms' in runs_df.columns:
    plot_df = runs_df[['params.temperature', 'metrics.latency_ms',
                        'metrics.response_length']].dropna()
    plot_df['params.temperature'] = plot_df['params.temperature'].astype(float)
    plot_df = plot_df.sort_values('params.temperature')

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].bar(plot_df['params.temperature'].astype(str),
                plot_df['metrics.latency_ms'], color='steelblue')
    axes[0].set_xlabel('Temperature'); axes[0].set_ylabel('Latency (ms)')
    axes[0].set_title('Latency by Temperature')

    axes[1].bar(plot_df['params.temperature'].astype(str),
                plot_df['metrics.response_length'], color='darkorange')
    axes[1].set_xlabel('Temperature'); axes[1].set_ylabel('Response Length (chars)')
    axes[1].set_title('Response Length by Temperature')

    plt.suptitle(f'Nemotron Temperature Sweep — {MODEL_ID}', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Run the sweep cells above first.')

## Summary

| Feature | API |
|---------|-----|
| Traced LLM call | `@mlflow.trace(span_type=SpanType.LLM)` |
| Token + latency attributes | `span.set_attribute('llm.total_tokens', ...)` |
| Multi-turn session grouping | `span.set_attribute('mlflow.trace.session_id', ...)` |
| Parameter sweep | One `mlflow.start_run()` per temperature value |
| Query and plot | `mlflow.search_runs()` + matplotlib |

**Dashboard:** Set experiment workflow mode to **GenAI** to see the Traces and Sessions tabs.

**Next:** `07_nemotron_rag_eval.ipynb` — RAG pipeline with systematic evaluation.